#### In this notebook, I am going to use an ANN model to predict classes. 
##### I am using the Digit recognizer data from Kaggle data. Although this dataset is generally used for CNN problems, I am using it to show how powerful ANNs can be.

In [2]:
import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

In [4]:
## ANN class defining all the layers
class ANNmodel(nn.Module):
    def __init__(self, input_layers, hidden_layers, output_layers):
        super(ANNmodel, self).__init__()

        self.f1 = nn.Linear(input_layers, hidden_layers)
        self.relu1 = nn.ReLU()

        self.f2 = nn.Linear(hidden_layers, hidden_layers)
        self.tanh1 = nn.Tanh()

        self.f3 = nn.Linear(hidden_layers, hidden_layers)
        self.elu1 = nn.ELU()

        self.f4 = nn.Linear(hidden_layers, output_layers)

    def forward(self, x):
        out = self.f1(x)
        out = self.relu1(out)

        out = self.f2(out)
        out = self.tanh1(out)

        out = self.f3(out)
        out = self.elu1(out)

        out = self.f4(out)

        return out

In [6]:
## Digit recognizer dataset from Kaggle
train = pd.read_csv(r"digit-recognizer/train.csv",dtype = np.float32)
valid = pd.read_csv(r"digit-recognizer/test.csv",dtype = np.float32)

In [10]:
## number of classes
len(train.label.unique())

10

In [13]:
## Splitting to training and test data
train_labels = train.label.values
train_features = train.loc[:, train.columns != 'label'].values

features_train, features_test, targets_train, targets_test = train_test_split(train_features, train_labels,
                                                                              test_size=0.25, random_state=123)

In [15]:
## Converting Training and test data to Tensors for Pytorch
featuresTrain = torch.from_numpy(features_train)
targetsTrain = torch.from_numpy(targets_train).type(torch.LongTensor)

featuresTest = torch.from_numpy(features_test)
targetsTest = torch.from_numpy(targets_test).type(torch.LongTensor)

In [17]:
## Defining batch size and number of iterations
batch_size = 100
num_iters = 10000
num_epochs = num_iters / (len(featuresTrain) / batch_size)
num_epochs = int(num_epochs)

In [19]:
## Setting up dataloaders for Pytorch
train = torch.utils.data.TensorDataset(featuresTrain, targetsTrain)
test = torch.utils.data.TensorDataset(featuresTest, targetsTest)

train_loader = DataLoader(train, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test, batch_size=batch_size, shuffle=False)

In [21]:
#Defining the number of layers
input_layers = 28 * 28
hidden_layers = 150
output_layers = 10

model = ANNmodel(input_layers, hidden_layers, output_layers)

In [23]:
## Defining crossentropy loss for multi class classification
loss_error = nn.CrossEntropyLoss()

## Stochastic gradient descent optimizer to find the optimal weights and biases
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)

In [25]:
count = 0
loss_list = []
iteration_list = []
accuracy_list = []

## Running the training and testing phase
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        train = Variable(images.view(-1, 28 * 28))
        labels = Variable(labels)

        optimizer.zero_grad()

        outputs = model(train)

        loss = loss_error(outputs, labels)

        loss.backward()

        optimizer.step()

        count = count + 1

        if count % 50 == 0:

            correct = 0
            total = 0

            for images, labels in test_loader:
                test = Variable(images.view(-1, 28 * 28))

                # Forward propagation
                outputs = model(test)

                # Get predictions from the maximum value
                predicted = torch.max(outputs.data, 1)[1]

                # Total number of labels
                total += len(labels)

                # Total correct predictions
                correct += (predicted == labels).sum()

            accuracy = 100 * correct / float(total)

            # store loss and iteration
            loss_list.append(loss.data)
            iteration_list.append(count)
            accuracy_list.append(accuracy)
        if count % 500 == 0:
            # Print Loss
            print('Iteration: {}  Loss: {}  Accuracy: {} %'.format(count, loss.data, accuracy))

Iteration: 500  Loss: 0.347804456949234  Accuracy: 93.29523468017578 %
Iteration: 1000  Loss: 0.19729959964752197  Accuracy: 94.74285888671875 %
Iteration: 1500  Loss: 0.047141171991825104  Accuracy: 95.60952758789062 %
Iteration: 2000  Loss: 0.09514220058917999  Accuracy: 95.68571472167969 %
Iteration: 2500  Loss: 0.07505393773317337  Accuracy: 96.29523468017578 %
Iteration: 3000  Loss: 0.04943738877773285  Accuracy: 96.4095230102539 %
Iteration: 3500  Loss: 0.04072361811995506  Accuracy: 96.65714263916016 %
Iteration: 4000  Loss: 0.011847127228975296  Accuracy: 96.26667022705078 %
Iteration: 4500  Loss: 0.029192175716161728  Accuracy: 96.75238037109375 %
Iteration: 5000  Loss: 0.07127992808818817  Accuracy: 96.80000305175781 %
Iteration: 5500  Loss: 0.02954024076461792  Accuracy: 96.68571472167969 %
Iteration: 6000  Loss: 0.08603975921869278  Accuracy: 96.92381286621094 %
Iteration: 6500  Loss: 0.013002977706491947  Accuracy: 96.95237731933594 %
Iteration: 7000  Loss: 0.1301482170820